# 🛡️ SleepGuard AI – Smart Classroom Analyzer
### Advanced Real-Time Student Attention & Sleep Detection System

---
**Features:**
- 👁️ Eye closure duration tracking (EAR - Eye Aspect Ratio)
- 🙇 Head tilt / pose estimation
- 📱 Phone usage detection
- 😮 Yawning frequency detection
- 📊 Attention score & Sleep probability %
- 🏆 Live distraction ranking dashboard

---

## 📦 CELL 1 — Install Dependencies

In [ ]:
# ============================================================
# CELL 1: Install all required packages
# ============================================================
import subprocess, sys

packages = [
    'opencv-python',
    'mediapipe',
    'numpy',
    'scipy',
    'matplotlib',
    'pandas',
    'playsound',
    'Pillow',
    'ipywidgets',
    'plotly'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed successfully!')

## 🔧 CELL 2 — Core Imports & Configuration

In [ ]:
# ============================================================
# CELL 2: Core Imports & Global Configuration
# ============================================================
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from scipy.spatial import distance as dist
from collections import deque, defaultdict
import time
import threading
import warnings
import math
warnings.filterwarnings('ignore')

# ─── CONFIGURATION ────────────────────────────────────────
CONFIG = {
    # Eye Aspect Ratio thresholds
    'EAR_THRESHOLD':       0.22,    # Below this = eyes closed
    'EAR_CONSEC_FRAMES':   20,      # Frames to confirm sleep

    # Mouth Aspect Ratio (yawn)
    'MAR_THRESHOLD':       0.65,    # Above this = yawning
    'YAWN_CONSEC_FRAMES':  15,

    # Head pose thresholds (degrees)
    'HEAD_PITCH_THRESHOLD': 15,     # Looking down
    'HEAD_YAW_THRESHOLD':   30,     # Looking sideways
    'HEAD_ROLL_THRESHOLD':  20,     # Head tilt (sleeping)

    # Scoring weights
    'WEIGHT_EYE':          0.35,
    'WEIGHT_HEAD':         0.30,
    'WEIGHT_YAWN':         0.20,
    'WEIGHT_PHONE':        0.15,

    # Alert thresholds
    'SLEEP_ALERT_SCORE':   0.75,
    'DISTRACTED_SCORE':    0.50,

    # History window (seconds)
    'HISTORY_WINDOW':      30,
    'FPS_TARGET':          30,

    # Display
    'FONT':                cv2.FONT_HERSHEY_SIMPLEX,
    'FONT_SCALE':          0.55,
    'LINE_HEIGHT':         22,
}

# ─── COLOR PALETTE ────────────────────────────────────────
COLORS = {
    'alert':      (0,   0,   255),   # Red
    'warning':    (0,   165, 255),   # Orange
    'ok':         (0,   255, 0),     # Green
    'info':       (255, 200, 0),     # Yellow
    'bg':         (20,  20,  20),    # Dark
    'panel':      (40,  40,  40),    # Panel
    'white':      (255, 255, 255),
    'cyan':       (255, 220, 0),
    'purple':     (255, 0,   200),
}

print('✅ Configuration loaded!')
print(f'   EAR Threshold    : {CONFIG["EAR_THRESHOLD"]}')
print(f'   MAR Threshold    : {CONFIG["MAR_THRESHOLD"]}')
print(f'   Head Pitch Limit : {CONFIG["HEAD_PITCH_THRESHOLD"]}°')
print(f'   Sleep Alert @    : {CONFIG["SLEEP_ALERT_SCORE"]*100:.0f}%')

## 👁️ CELL 3 — Eye & Mouth Aspect Ratio Engine

In [ ]:
# ============================================================
# CELL 3: EAR / MAR Calculation Engine
# ============================================================

# MediaPipe FaceMesh landmark indices
# Left eye
LEFT_EYE  = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
RIGHT_EYE = [33,  7,   163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]

# Simplified 6-point landmarks for EAR
LEFT_EYE_EAR  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE_EAR = [33,  160, 158, 133, 153, 144]

# Mouth landmarks for MAR (yawn detection)
MOUTH_EAR = [61, 291, 39, 181, 0, 17, 269, 405]


def eye_aspect_ratio(landmarks, eye_indices, img_w, img_h):
    """Calculate Eye Aspect Ratio (EAR) using 6-point method."""
    pts = []
    for idx in eye_indices:
        lm = landmarks[idx]
        pts.append((lm.x * img_w, lm.y * img_h))

    # Vertical distances
    A = dist.euclidean(pts[1], pts[5])
    B = dist.euclidean(pts[2], pts[4])
    # Horizontal distance
    C = dist.euclidean(pts[0], pts[3])

    ear = (A + B) / (2.0 * C + 1e-6)
    return ear


def mouth_aspect_ratio(landmarks, mouth_indices, img_w, img_h):
    """Calculate Mouth Aspect Ratio (MAR) for yawn detection."""
    pts = []
    for idx in mouth_indices:
        lm = landmarks[idx]
        pts.append((lm.x * img_w, lm.y * img_h))

    # Vertical
    A = dist.euclidean(pts[2], pts[6])
    B = dist.euclidean(pts[3], pts[5])
    # Horizontal
    C = dist.euclidean(pts[0], pts[4])

    mar = (A + B) / (2.0 * C + 1e-6)
    return mar


def get_landmark_point(landmarks, idx, img_w, img_h):
    """Convert normalized landmark to pixel coordinates."""
    lm = landmarks[idx]
    return int(lm.x * img_w), int(lm.y * img_h)


def draw_eye_landmarks(frame, landmarks, eye_indices, img_w, img_h, color):
    """Draw eye contour on frame."""
    pts = []
    for idx in eye_indices:
        x, y = get_landmark_point(landmarks, idx, img_w, img_h)
        pts.append([x, y])
    pts = np.array(pts, dtype=np.int32)
    cv2.polylines(frame, [pts], True, color, 1, cv2.LINE_AA)


print('✅ EAR/MAR engine loaded!')
print('   Using 6-point Eye Aspect Ratio formula')
print('   Using 8-point Mouth Aspect Ratio formula')

## 🧠 CELL 4 — Head Pose Estimation

In [ ]:
# ============================================================
# CELL 4: Head Pose Estimation (Pitch / Yaw / Roll)
# ============================================================

# 3D model reference points for pose estimation
MODEL_POINTS_3D = np.array([
    (0.0,    0.0,    0.0),     # Nose tip
    (0.0,   -330.0, -65.0),    # Chin
    (-225.0, 170.0, -135.0),   # Left eye left corner
    (225.0,  170.0, -135.0),   # Right eye right corner
    (-150.0,-150.0, -125.0),   # Left mouth corner
    (150.0, -150.0, -125.0),   # Right mouth corner
], dtype=np.float64)

# MediaPipe landmark indices for pose estimation
POSE_LANDMARK_IDS = [1, 152, 226, 446, 57, 287]


class HeadPoseEstimator:
    def __init__(self, img_w, img_h):
        focal_length = img_w
        center = (img_w / 2, img_h / 2)
        self.camera_matrix = np.array([
            [focal_length, 0,            center[0]],
            [0,            focal_length, center[1]],
            [0,            0,            1]
        ], dtype=np.float64)
        self.dist_coeffs = np.zeros((4, 1))
        self.img_w = img_w
        self.img_h = img_h

    def get_pose(self, landmarks):
        """Estimate head pose from face landmarks.
        Returns (pitch, yaw, roll) in degrees."""
        image_points = []
        for idx in POSE_LANDMARK_IDS:
            lm = landmarks[idx]
            x = lm.x * self.img_w
            y = lm.y * self.img_h
            image_points.append((x, y))
        image_points = np.array(image_points, dtype=np.float64)

        success, rot_vec, trans_vec = cv2.solvePnP(
            MODEL_POINTS_3D,
            image_points,
            self.camera_matrix,
            self.dist_coeffs,
            flags=cv2.SOLVEPNP_ITERATIVE
        )

        if not success:
            return 0.0, 0.0, 0.0

        rot_mat, _ = cv2.Rodrigues(rot_vec)
        pose_mat = cv2.hconcat([rot_mat, trans_vec])
        _, _, _, _, _, _, euler = cv2.decomposeProjectionMatrix(
            cv2.hconcat([pose_mat, np.array([[0],[0],[0],[1]], dtype=np.float64).T])
        )

        pitch = euler[0][0]
        yaw   = euler[1][0]
        roll  = euler[2][0]
        return float(pitch), float(yaw), float(roll)

    def draw_axis(self, frame, landmarks, rot_vec=None, trans_vec=None):
        """Draw 3D axes on face for visualization."""
        nose_tip = get_landmark_point(landmarks, 1, self.img_w, self.img_h)
        # Draw simple direction indicator
        cv2.circle(frame, nose_tip, 4, COLORS['cyan'], -1)


print('✅ Head Pose Estimator loaded!')
print('   Estimates: Pitch (nod), Yaw (shake), Roll (tilt)')
print('   Using solvePnP with 6-point facial model')

## 📱 CELL 5 — Phone Detection Module

In [ ]:
# ============================================================
# CELL 5: Phone Usage Detection (Hand + Object heuristic)
# ============================================================

class PhoneDetector:
    """Detects phone usage via hand position analysis.
    Uses MediaPipe Hands to detect raised hands near face level
    in a phone-holding posture."""

    def __init__(self):
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.6,
            min_tracking_confidence=0.5
        )
        self.phone_frames = 0
        self.phone_detected = False
        self.PHONE_CONSEC = 10

    def detect(self, frame_rgb, face_bbox=None):
        """Detect phone usage from frame.
        Returns (is_phone, confidence, hand_landmarks)."""
        results = self.hands.process(frame_rgb)
        hand_lms_list = []

        if not results.multi_hand_landmarks:
            self.phone_frames = max(0, self.phone_frames - 1)
            if self.phone_frames == 0:
                self.phone_detected = False
            return self.phone_detected, 0.0, []

        h, w = frame_rgb.shape[:2]
        phone_score = 0.0

        for hand_lms in results.multi_hand_landmarks:
            hand_lms_list.append(hand_lms)

            # Get wrist and fingertip positions
            wrist      = hand_lms.landmark[0]
            index_tip  = hand_lms.landmark[8]
            thumb_tip  = hand_lms.landmark[4]
            pinky_tip  = hand_lms.landmark[20]

            # Phone-holding heuristic:
            # 1. Hand is in upper portion of frame (near face)
            # 2. Fingers are relatively vertical/grouped
            # 3. Hand position is lateral (not centered over keyboard)

            score = 0.0

            # Hand in upper 60% of frame
            if wrist.y < 0.6:
                score += 0.3

            # Fingers somewhat extended (phone grip)
            finger_spread = abs(thumb_tip.x - pinky_tip.x)
            if 0.05 < finger_spread < 0.25:
                score += 0.3

            # Hand raised near face level
            if wrist.y < 0.45:
                score += 0.2

            # Vertical orientation typical of phone holding
            if face_bbox:
                fx, fy, fw, fh = face_bbox
                hand_x_norm = wrist.x
                # Hand within or near face X range
                face_left  = fx / w
                face_right = (fx + fw) / w
                if face_left - 0.15 < hand_x_norm < face_right + 0.15:
                    score += 0.2

            phone_score = max(phone_score, score)

        if phone_score > 0.55:
            self.phone_frames = min(self.phone_frames + 2, self.PHONE_CONSEC * 2)
        else:
            self.phone_frames = max(0, self.phone_frames - 1)

        self.phone_detected = self.phone_frames >= self.PHONE_CONSEC
        return self.phone_detected, phone_score, hand_lms_list

    def draw_hands(self, frame, hand_lms_list, is_phone):
        mp_drawing = mp.solutions.drawing_utils
        h, w = frame.shape[:2]
        color = COLORS['alert'] if is_phone else COLORS['ok']
        style = mp_drawing.DrawingSpec(color=color, thickness=2, circle_radius=3)
        for hand_lms in hand_lms_list:
            mp_drawing.draw_landmarks(
                frame, hand_lms,
                mp.solutions.hands.HAND_CONNECTIONS,
                style, style
            )


print('✅ Phone Detection module loaded!')
print('   Method: Hand pose + position heuristic analysis')
print('   Detection: Hand position, finger spread, face proximity')

## 📊 CELL 6 — Student State Tracker & Scoring Engine

In [ ]:
# ============================================================
# CELL 6: Student State Tracker & Attention Scoring Engine
# ============================================================

class StudentTracker:
    """Tracks a single student's attention state over time."""

    def __init__(self, student_id='Student_1', history_sec=30, fps=30):
        self.id = student_id
        maxlen = history_sec * fps

        # Rolling history buffers
        self.ear_history    = deque(maxlen=maxlen)
        self.mar_history    = deque(maxlen=maxlen)
        self.pitch_history  = deque(maxlen=maxlen)
        self.yaw_history    = deque(maxlen=maxlen)
        self.roll_history   = deque(maxlen=maxlen)
        self.phone_history  = deque(maxlen=maxlen)
        self.score_history  = deque(maxlen=maxlen)
        self.sleep_history  = deque(maxlen=maxlen)
        self.time_history   = deque(maxlen=maxlen)

        # Frame counters
        self.eye_closed_frames = 0
        self.yawn_frames       = 0
        self.head_down_frames  = 0

        # Event counters
        self.total_blinks    = 0
        self.total_yawns     = 0
        self.sleep_episodes  = 0
        self.phone_episodes  = 0

        # State flags
        self.is_sleeping     = False
        self.is_yawning      = False
        self.is_phone        = False
        self.is_head_down    = False
        self.prev_sleeping   = False
        self.prev_phone      = False

        # Timestamps
        self.sleep_start     = None
        self.total_sleep_sec = 0.0
        self.session_start   = time.time()

        # Current computed metrics
        self.current = {
            'ear': 0.0, 'mar': 0.0,
            'pitch': 0.0, 'yaw': 0.0, 'roll': 0.0,
            'attention_score': 100.0,
            'sleep_probability': 0.0,
            'distraction_level': 'ATTENTIVE',
            'alert_color': COLORS['ok']
        }

    def update(self, ear, mar, pitch, yaw, roll, phone_detected):
        """Update tracker with new frame measurements."""
        t = time.time()

        # ── Eye state ─────────────────────────────────────
        if ear < CONFIG['EAR_THRESHOLD']:
            self.eye_closed_frames += 1
        else:
            if self.eye_closed_frames >= 3:
                self.total_blinks += 1
            self.eye_closed_frames = 0

        was_sleeping = self.is_sleeping
        self.is_sleeping = self.eye_closed_frames >= CONFIG['EAR_CONSEC_FRAMES']
        if self.is_sleeping and not was_sleeping:
            self.sleep_episodes += 1
            self.sleep_start = t
        if not self.is_sleeping and was_sleeping and self.sleep_start:
            self.total_sleep_sec += t - self.sleep_start
            self.sleep_start = None

        # ── Yawn state ────────────────────────────────────
        if mar > CONFIG['MAR_THRESHOLD']:
            self.yawn_frames += 1
        else:
            if self.yawn_frames >= CONFIG['YAWN_CONSEC_FRAMES']:
                self.total_yawns += 1
            self.yawn_frames = 0
        self.is_yawning = self.yawn_frames >= CONFIG['YAWN_CONSEC_FRAMES']

        # ── Head state ────────────────────────────────────
        head_alert = (
            abs(pitch) > CONFIG['HEAD_PITCH_THRESHOLD'] or
            abs(yaw)   > CONFIG['HEAD_YAW_THRESHOLD']   or
            abs(roll)  > CONFIG['HEAD_ROLL_THRESHOLD']
        )
        self.is_head_down = head_alert

        # ── Phone state ───────────────────────────────────
        if phone_detected and not self.prev_phone:
            self.phone_episodes += 1
        self.is_phone = phone_detected
        self.prev_phone = phone_detected

        # ── Compute scores ────────────────────────────────
        eye_score  = self._eye_score(ear)
        head_score = self._head_score(pitch, yaw, roll)
        yawn_score = self._yawn_score()
        phone_score= 1.0 if phone_detected else 0.0

        distraction = (
            CONFIG['WEIGHT_EYE']   * eye_score   +
            CONFIG['WEIGHT_HEAD']  * head_score  +
            CONFIG['WEIGHT_YAWN']  * yawn_score  +
            CONFIG['WEIGHT_PHONE'] * phone_score
        )
        distraction = np.clip(distraction, 0.0, 1.0)

        # Sleep probability (weighted differently)
        sleep_prob = (
            0.50 * eye_score +
            0.30 * head_score +
            0.20 * yawn_score
        )
        sleep_prob = np.clip(sleep_prob, 0.0, 1.0)

        attention = 100.0 * (1.0 - distraction)

        # Determine label
        if sleep_prob > 0.75 or self.is_sleeping:
            level = 'SLEEPING'
            color = COLORS['alert']
        elif distraction > 0.60:
            level = 'DISTRACTED'
            color = COLORS['warning']
        elif distraction > 0.35:
            level = 'DROWSY'
            color = COLORS['info']
        else:
            level = 'ATTENTIVE'
            color = COLORS['ok']

        # Store current values
        self.current.update({
            'ear': ear, 'mar': mar,
            'pitch': pitch, 'yaw': yaw, 'roll': roll,
            'attention_score': attention,
            'sleep_probability': sleep_prob * 100,
            'distraction_level': level,
            'alert_color': color,
        })

        # Append to history
        self.ear_history.append(ear)
        self.mar_history.append(mar)
        self.pitch_history.append(pitch)
        self.yaw_history.append(yaw)
        self.roll_history.append(roll)
        self.phone_history.append(float(phone_detected))
        self.score_history.append(attention)
        self.sleep_history.append(sleep_prob * 100)
        self.time_history.append(t - self.session_start)

    def _eye_score(self, ear):
        """Normalize EAR to [0,1] distraction score."""
        # EAR: normal ~0.3, closed ~0.15
        t = CONFIG['EAR_THRESHOLD']
        if ear >= t + 0.05:
            return 0.0
        elif ear <= t - 0.05:
            return 1.0
        return 1.0 - (ear - (t - 0.05)) / 0.10

    def _head_score(self, pitch, yaw, roll):
        """Normalize head angles to distraction score."""
        p_score = min(abs(pitch) / CONFIG['HEAD_PITCH_THRESHOLD'], 1.0)
        y_score = min(abs(yaw)   / CONFIG['HEAD_YAW_THRESHOLD'],   1.0)
        r_score = min(abs(roll)  / CONFIG['HEAD_ROLL_THRESHOLD'],  1.0)
        return max(p_score, y_score, r_score)

    def _yawn_score(self):
        """Yawn score based on recent yawn frames."""
        ratio = self.yawn_frames / CONFIG['YAWN_CONSEC_FRAMES']
        return min(ratio, 1.0)

    def session_summary(self):
        """Return session statistics."""
        elapsed = time.time() - self.session_start
        avg_att = np.mean(list(self.score_history)) if self.score_history else 100.0
        avg_sleep = np.mean(list(self.sleep_history)) if self.sleep_history else 0.0
        return {
            'student_id':      self.id,
            'session_sec':     elapsed,
            'avg_attention':   round(avg_att, 1),
            'avg_sleep_prob':  round(avg_sleep, 1),
            'total_blinks':    self.total_blinks,
            'total_yawns':     self.total_yawns,
            'sleep_episodes':  self.sleep_episodes,
            'phone_episodes':  self.phone_episodes,
            'total_sleep_sec': round(self.total_sleep_sec, 1),
        }


print('✅ StudentTracker engine loaded!')
print('   Tracks: EAR, MAR, Head pose, Phone, Attention score, Sleep probability')

## 🖥️ CELL 7 — HUD / Overlay Renderer

In [ ]:
# ============================================================
# CELL 7: Advanced HUD Overlay Renderer
# ============================================================

def draw_rounded_rect(img, pt1, pt2, color, radius=8, thickness=-1, alpha=0.7):
    """Draw a semi-transparent rounded rectangle."""
    overlay = img.copy()
    x1, y1 = pt1
    x2, y2 = pt2
    cv2.rectangle(overlay, (x1+radius, y1), (x2-radius, y2), color, thickness)
    cv2.rectangle(overlay, (x1, y1+radius), (x2, y2-radius), color, thickness)
    cv2.circle(overlay, (x1+radius, y1+radius), radius, color, thickness)
    cv2.circle(overlay, (x2-radius, y1+radius), radius, color, thickness)
    cv2.circle(overlay, (x1+radius, y2-radius), radius, color, thickness)
    cv2.circle(overlay, (x2-radius, y2-radius), radius, color, thickness)
    cv2.addWeighted(overlay, alpha, img, 1-alpha, 0, img)


def draw_bar(frame, x, y, w, h, value, max_val, color_low, color_high, label=''):
    """Draw a horizontal progress bar."""
    # Background
    cv2.rectangle(frame, (x, y), (x+w, y+h), (60,60,60), -1)
    # Fill
    ratio = np.clip(value / max_val, 0, 1)
    fill_w = int(w * ratio)
    # Color gradient based on ratio
    r = int(color_high[2] * ratio + color_low[2] * (1-ratio))
    g = int(color_high[1] * ratio + color_low[1] * (1-ratio))
    b = int(color_high[0] * ratio + color_low[0] * (1-ratio))
    if fill_w > 0:
        cv2.rectangle(frame, (x, y), (x+fill_w, y+h), (b,g,r), -1)
    # Border
    cv2.rectangle(frame, (x, y), (x+w, y+h), (100,100,100), 1)
    # Label
    if label:
        cv2.putText(frame, label, (x, y-4),
                    CONFIG['FONT'], 0.38, COLORS['white'], 1, cv2.LINE_AA)
    return ratio


def draw_circular_gauge(frame, cx, cy, radius, value, max_val,
                         color, label='', unit='%'):
    """Draw a circular arc gauge."""
    # Background arc
    cv2.ellipse(frame, (cx, cy), (radius, radius), -90, 0, 270,
                (60,60,60), 4)
    # Colored arc
    angle = int(270 * np.clip(value / max_val, 0, 1))
    if angle > 0:
        cv2.ellipse(frame, (cx, cy), (radius, radius), -90, 0, angle,
                    color, 4)
    # Center text
    val_str = f'{value:.0f}{unit}'
    ts = cv2.getTextSize(val_str, CONFIG['FONT'], 0.5, 1)[0]
    cv2.putText(frame, val_str, (cx - ts[0]//2, cy + 5),
                CONFIG['FONT'], 0.5, COLORS['white'], 1, cv2.LINE_AA)
    # Label below
    ts2 = cv2.getTextSize(label, CONFIG['FONT'], 0.38, 1)[0]
    cv2.putText(frame, label, (cx - ts2[0]//2, cy + radius + 14),
                CONFIG['FONT'], 0.38, COLORS['white'], 1, cv2.LINE_AA)


class HUDRenderer:
    """Renders the full HUD overlay on the video frame."""

    def __init__(self, frame_w, frame_h):
        self.fw = frame_w
        self.fh = frame_h

    def render(self, frame, tracker, face_bbox=None, fps=0):
        """Draw complete HUD on frame."""
        c = tracker.current

        # ── Top Status Banner ──────────────────────────────
        banner_color = c['alert_color']
        draw_rounded_rect(frame, (0, 0), (self.fw, 40),
                          (banner_color[0]//3, banner_color[1]//3, banner_color[2]//3),
                          radius=0, alpha=0.85)
        status_text = f"  🛡 SleepGuard AI  |  {c['distraction_level']}  |  FPS: {fps:.0f}"
        cv2.putText(frame, f"SleepGuard AI  |  {c['distraction_level']}  |  FPS: {fps:.0f}",
                    (10, 27), CONFIG['FONT'], 0.65, COLORS['white'], 2, cv2.LINE_AA)

        # ── Left Panel ────────────────────────────────────
        px, py = 10, 55
        pw, ph = 200, 280
        draw_rounded_rect(frame, (px-5, py-5), (px+pw, py+ph),
                          (30,30,30), alpha=0.80)

        # Title
        cv2.putText(frame, 'METRICS', (px, py+14),
                    CONFIG['FONT'], 0.50, COLORS['cyan'], 1, cv2.LINE_AA)
        cv2.line(frame, (px, py+20), (px+pw-10, py+20), (80,80,80), 1)

        # EAR Bar
        ear_color = COLORS['alert'] if c['ear'] < CONFIG['EAR_THRESHOLD'] else COLORS['ok']
        draw_bar(frame, px, py+35, pw-10, 12, c['ear'], 0.45,
                 (0,0,255), (0,255,0), f"EAR: {c['ear']:.3f}")
        # Threshold line
        thresh_x = px + int((pw-10) * CONFIG['EAR_THRESHOLD'] / 0.45)
        cv2.line(frame, (thresh_x, py+35), (thresh_x, py+47), COLORS['info'], 2)

        # MAR Bar
        draw_bar(frame, px, py+72, pw-10, 12, c['mar'], 1.2,
                 (0,200,200), (0,50,255), f"MAR: {c['mar']:.3f}")
        thresh_x2 = px + int((pw-10) * CONFIG['MAR_THRESHOLD'] / 1.2)
        cv2.line(frame, (thresh_x2, py+72), (thresh_x2, py+84), COLORS['info'], 2)

        # Head Pose
        y_off = py + 104
        cv2.putText(frame, 'HEAD POSE:', (px, y_off),
                    CONFIG['FONT'], 0.40, COLORS['white'], 1, cv2.LINE_AA)
        y_off += 16
        for label, val, limit in [
            (f'Pitch {c["pitch"]:+.1f}', abs(c['pitch']), CONFIG['HEAD_PITCH_THRESHOLD']),
            (f'Yaw   {c["yaw"]:+.1f}',   abs(c['yaw']),   CONFIG['HEAD_YAW_THRESHOLD']),
            (f'Roll  {c["roll"]:+.1f}',   abs(c['roll']),  CONFIG['HEAD_ROLL_THRESHOLD']),
        ]:
            exceeded = val > limit
            col = COLORS['alert'] if exceeded else COLORS['ok']
            draw_bar(frame, px, y_off, pw-10, 10, val, limit * 1.5,
                     (0,180,0), (0,0,200), label)
            y_off += 28

        # Event counters
        y_off += 5
        for label, val in [
            (f'Blinks: {tracker.total_blinks}', None),
            (f'Yawns:  {tracker.total_yawns}',  None),
            (f'Sleep episodes: {tracker.sleep_episodes}', None),
            (f'Phone episodes: {tracker.phone_episodes}', None),
        ]:
            cv2.putText(frame, label, (px, y_off),
                        CONFIG['FONT'], 0.40, COLORS['white'], 1, cv2.LINE_AA)
            y_off += 16

        # ── Right Panel — Gauges ──────────────────────────
        rpx = self.fw - 175
        rpy = 55
        draw_rounded_rect(frame, (rpx-10, rpy-5), (self.fw-5, rpy+280),
                          (30,30,30), alpha=0.80)

        cv2.putText(frame, 'SCORES', (rpx, rpy+14),
                    CONFIG['FONT'], 0.50, COLORS['cyan'], 1, cv2.LINE_AA)
        cv2.line(frame, (rpx, rpy+20), (self.fw-10, rpy+20), (80,80,80), 1)

        # Attention gauge
        att = c['attention_score']
        att_color = COLORS['ok'] if att > 70 else (COLORS['warning'] if att > 40 else COLORS['alert'])
        draw_circular_gauge(frame, rpx+75, rpy+75, 45,
                            att, 100, att_color, 'Attention', '%')

        # Sleep probability gauge
        slp = c['sleep_probability']
        slp_color = COLORS['alert'] if slp > 75 else (COLORS['warning'] if slp > 40 else COLORS['ok'])
        draw_circular_gauge(frame, rpx+75, rpy+175, 45,
                            slp, 100, slp_color, 'Sleep Prob', '%')

        # ── Alert Badges ──────────────────────────────────
        badges = []
        if tracker.is_sleeping:
            badges.append(('SLEEPING!',   COLORS['alert']))
        if tracker.is_yawning:
            badges.append(('YAWNING',     COLORS['warning']))
        if tracker.is_head_down:
            badges.append(('HEAD DOWN',   COLORS['warning']))
        if tracker.is_phone:
            badges.append(('PHONE USE!',  COLORS['purple']))

        badge_y = self.fh - 50
        badge_x = (self.fw - sum(len(b[0])*12 + 20 for b in badges)) // 2 if badges else 0
        for btxt, bcol in badges:
            bw = len(btxt) * 11 + 16
            draw_rounded_rect(frame, (badge_x, badge_y),
                              (badge_x+bw, badge_y+28), bcol, alpha=0.9)
            cv2.putText(frame, btxt, (badge_x+8, badge_y+19),
                        CONFIG['FONT'], 0.55, COLORS['white'], 2, cv2.LINE_AA)
            badge_x += bw + 10

        # ── Face bounding box ─────────────────────────────
        if face_bbox:
            fx, fy, fw, fh = face_bbox
            box_color = c['alert_color']
            cv2.rectangle(frame, (fx, fy), (fx+fw, fy+fh), box_color, 2)
            # Corner accents
            cs = 15
            for cx_, cy_, dx, dy in [
                (fx, fy, 1, 1), (fx+fw, fy, -1, 1),
                (fx, fy+fh, 1, -1), (fx+fw, fy+fh, -1, -1)
            ]:
                cv2.line(frame, (cx_, cy_), (cx_+dx*cs, cy_), box_color, 3)
                cv2.line(frame, (cx_, cy_), (cx_, cy_+dy*cs), box_color, 3)

        # ── Session time ──────────────────────────────────
        elapsed = time.time() - tracker.session_start
        m, s = divmod(int(elapsed), 60)
        cv2.putText(frame, f'Session: {m:02d}:{s:02d}  |  '
                    f'Sleep time: {tracker.total_sleep_sec:.1f}s',
                    (10, self.fh - 12),
                    CONFIG['FONT'], 0.42, COLORS['white'], 1, cv2.LINE_AA)

        return frame


print('✅ HUD Renderer loaded!')
print('   Renders: Metric bars, Circular gauges, Alert badges, Face box, Pose indicators')

## 🏫 CELL 8 — Classroom-Level Ranking Dashboard

In [ ]:
# ============================================================
# CELL 8: Multi-Student Classroom Ranking System
# ============================================================

class ClassroomDashboard:
    """Simulates tracking and ranking multiple students."""

    def __init__(self, num_students=6):
        self.students = {
            f'Student_{i+1}': StudentTracker(student_id=f'Student_{i+1}')
            for i in range(num_students)
        }
        self.num_students = num_students

    def update_student(self, student_id, ear, mar, pitch, yaw, roll, phone):
        if student_id in self.students:
            self.students[student_id].update(ear, mar, pitch, yaw, roll, phone)

    def get_ranking(self):
        """Return students sorted by attention score (worst first)."""
        summaries = [t.session_summary() for t in self.students.values()]
        summaries.sort(key=lambda x: x['avg_attention'])
        return summaries

    def plot_dashboard(self):
        """Plot the classroom dashboard with matplotlib."""
        ranking = self.get_ranking()
        names   = [r['student_id'] for r in ranking]
        att     = [r['avg_attention'] for r in ranking]
        sleep_p = [r['avg_sleep_prob'] for r in ranking]
        yawns   = [r['total_yawns'] for r in ranking]
        phones  = [r['phone_episodes'] for r in ranking]

        colors = []
        for a in att:
            if a < 40:   colors.append('#FF4444')
            elif a < 65: colors.append('#FF8800')
            elif a < 80: colors.append('#FFCC00')
            else:        colors.append('#44CC44')

        fig = plt.figure(figsize=(16, 10), facecolor='#1a1a2e')
        fig.suptitle('🛡️ SleepGuard AI — Classroom Attention Dashboard',
                     color='white', fontsize=16, fontweight='bold', y=0.98)

        gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35,
                              left=0.08, right=0.96, top=0.92, bottom=0.08)

        # ── Plot 1: Attention Ranking Bar Chart ──────────
        ax1 = fig.add_subplot(gs[0, :2])
        ax1.set_facecolor('#16213e')
        bars = ax1.barh(names, att, color=colors, edgecolor='white',
                        linewidth=0.5, height=0.6)
        ax1.set_xlim(0, 100)
        ax1.axvline(70, color='#FFCC00', linestyle='--', linewidth=1.5, alpha=0.8)
        ax1.axvline(40, color='#FF4444', linestyle='--', linewidth=1.5, alpha=0.8)
        ax1.text(71, -0.7, 'Drowsy', color='#FFCC00', fontsize=8)
        ax1.text(41, -0.7, 'Critical', color='#FF4444', fontsize=8)
        for bar, val in zip(bars, att):
            ax1.text(min(val+1, 97), bar.get_y()+bar.get_height()/2,
                     f'{val:.1f}%', va='center', color='white', fontsize=9)
        ax1.set_xlabel('Attention Score (%)', color='#aaaaaa')
        ax1.set_title('📊 Distraction Ranking (Worst → Best)', color='white', fontsize=12)
        ax1.tick_params(colors='white')
        ax1.spines[['top','right','bottom','left']].set_color('#444')
        # Rank labels
        for i, name in enumerate(names):
            medal = ['🥇','🥈','🥉'][i] if i < 3 else f'#{i+1}'
            ax1.text(-2, i, medal, ha='right', va='center',
                     color='white', fontsize=10)

        # ── Plot 2: Sleep Probability ─────────────────────
        ax2 = fig.add_subplot(gs[0, 2])
        ax2.set_facecolor('#16213e')
        sleep_colors = ['#FF4444' if s > 60 else '#FF8800' if s > 30 else '#44CC44'
                        for s in sleep_p]
        wedges, texts, autotexts = ax2.pie(
            [max(s, 1) for s in sleep_p],
            labels=names, autopct='%1.0f%%',
            colors=sleep_colors,
            startangle=90,
            textprops={'color': 'white', 'fontsize': 8}
        )
        ax2.set_title('😴 Sleep Probability', color='white', fontsize=12)

        # ── Plot 3: Yawn Count ────────────────────────────
        ax3 = fig.add_subplot(gs[1, 0])
        ax3.set_facecolor('#16213e')
        yawn_colors = ['#FF4444' if y > 5 else '#FF8800' if y > 2 else '#44CC44'
                       for y in yawns]
        ax3.bar(names, yawns, color=yawn_colors, edgecolor='white', linewidth=0.5)
        ax3.set_title('😮 Yawn Count', color='white', fontsize=12)
        ax3.set_ylabel('Count', color='#aaaaaa')
        ax3.tick_params(colors='white', axis='both')
        ax3.tick_params(axis='x', rotation=45)
        ax3.spines[['top','right']].set_color('#444')
        for rect, val in zip(ax3.patches, yawns):
            ax3.text(rect.get_x()+rect.get_width()/2, rect.get_height()+0.05,
                     str(val), ha='center', color='white', fontsize=9)

        # ── Plot 4: Phone Usage ───────────────────────────
        ax4 = fig.add_subplot(gs[1, 1])
        ax4.set_facecolor('#16213e')
        phone_colors = ['#FF4444' if p > 3 else '#FF8800' if p > 1 else '#44CC44'
                        for p in phones]
        ax4.bar(names, phones, color=phone_colors, edgecolor='white', linewidth=0.5)
        ax4.set_title('📱 Phone Usage Events', color='white', fontsize=12)
        ax4.set_ylabel('Episodes', color='#aaaaaa')
        ax4.tick_params(colors='white', axis='both')
        ax4.tick_params(axis='x', rotation=45)
        ax4.spines[['top','right']].set_color('#444')

        # ── Plot 5: Legend / Summary ──────────────────────
        ax5 = fig.add_subplot(gs[1, 2])
        ax5.set_facecolor('#16213e')
        ax5.axis('off')

        legend_data = [
            ('🟢 ATTENTIVE',   '80–100%',  '#44CC44'),
            ('🟡 DROWSY',      '65–80%',   '#FFCC00'),
            ('🟠 DISTRACTED',  '40–65%',   '#FF8800'),
            ('🔴 SLEEPING',    '0–40%',    '#FF4444'),
        ]
        ax5.text(0.5, 0.95, 'Attention Legend', ha='center',
                 color='white', fontsize=11, fontweight='bold',
                 transform=ax5.transAxes)
        for i, (status, rng, col) in enumerate(legend_data):
            y = 0.80 - i * 0.16
            ax5.add_patch(mpatches.FancyBboxPatch(
                (0.05, y-0.04), 0.90, 0.12,
                boxstyle='round,pad=0.01',
                facecolor=col, alpha=0.3,
                transform=ax5.transAxes
            ))
            ax5.text(0.10, y+0.02, status, color=col, fontsize=9,
                     transform=ax5.transAxes)
            ax5.text(0.80, y+0.02, rng, color='white', fontsize=9,
                     transform=ax5.transAxes, ha='right')

        plt.savefig('/tmp/classroom_dashboard.png', dpi=120,
                    bbox_inches='tight', facecolor='#1a1a2e')
        plt.show()
        print('✅ Dashboard saved to /tmp/classroom_dashboard.png')


print('✅ ClassroomDashboard loaded!')
print('   Tracks up to N students with ranking, yawns, phone usage, sleep probability')

## 🎮 CELL 9 — Main Detection Loop

In [ ]:
# ============================================================
# CELL 9: Main SleepGuard AI Detection Loop
# ============================================================
# Press 'Q' to quit, 'D' to show dashboard, 'R' to reset stats

def run_sleepguard(camera_index=0, demo_mode=False):
    """
    Main entry point for SleepGuard AI.
    
    Args:
        camera_index: Webcam index (0 = default)
        demo_mode:    If True, uses synthetic data (no webcam required)
    """

    # ── Initialize MediaPipe ──────────────────────────────
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    mp_face_det = mp.solutions.face_detection
    face_det = mp_face_det.FaceDetection(min_detection_confidence=0.5)

    phone_detector = PhoneDetector()
    tracker = StudentTracker('Student_1')

    # ── Open camera ───────────────────────────────────────
    if not demo_mode:
        cap = cv2.VideoCapture(camera_index)
        if not cap.isOpened():
            print('❌ Camera not found. Switching to DEMO MODE...')
            demo_mode = True
        else:
            cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
            cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
            cap.set(cv2.CAP_PROP_FPS, 30)
            W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            hud = HUDRenderer(W, H)
            head_pose = HeadPoseEstimator(W, H)
            print(f'✅ Camera opened: {W}x{H}')

    if demo_mode:
        W, H = 1280, 720
        hud = HUDRenderer(W, H)
        print('🎮 Running in DEMO MODE — synthetic data simulation')
        _run_demo(tracker, hud)
        return

    # ── FPS tracking ──────────────────────────────────────
    fps_counter = 0
    fps_display = 0
    fps_timer = time.time()

    print('🚀 SleepGuard AI running...')
    print('   Controls: Q=Quit | D=Dashboard | R=Reset stats | S=Screenshot')

    while True:
        ret, frame = cap.read()
        if not ret:
            print('❌ Frame capture failed!')
            break

        frame = cv2.flip(frame, 1)  # Mirror
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w = frame.shape[:2]

        # ── Default values ────────────────────────────────
        ear = 0.30
        mar = 0.30
        pitch = yaw = roll = 0.0
        face_bbox = None

        # ── Face Mesh ─────────────────────────────────────
        mesh_results = face_mesh.process(frame_rgb)

        if mesh_results.multi_face_landmarks:
            lms = mesh_results.multi_face_landmarks[0].landmark

            # EAR
            left_ear  = eye_aspect_ratio(lms, LEFT_EYE_EAR,  w, h)
            right_ear = eye_aspect_ratio(lms, RIGHT_EYE_EAR, w, h)
            ear = (left_ear + right_ear) / 2.0

            # MAR
            mar = mouth_aspect_ratio(lms, MOUTH_EAR, w, h)

            # Draw eye contours
            eye_color = COLORS['alert'] if ear < CONFIG['EAR_THRESHOLD'] else COLORS['ok']
            draw_eye_landmarks(frame, lms, LEFT_EYE_EAR,  w, h, eye_color)
            draw_eye_landmarks(frame, lms, RIGHT_EYE_EAR, w, h, eye_color)

            # Head pose
            try:
                pitch, yaw, roll = head_pose.get_pose(lms)
            except:
                pass

            # Face bounding box from landmarks
            xs = [int(lm.x * w) for lm in lms]
            ys = [int(lm.y * h) for lm in lms]
            x1, y1 = max(min(xs)-10, 0), max(min(ys)-10, 0)
            x2, y2 = min(max(xs)+10, w), min(max(ys)+10, h)
            face_bbox = (x1, y1, x2-x1, y2-y1)

        # ── Phone Detection ───────────────────────────────
        is_phone, phone_conf, hand_lms = phone_detector.detect(frame_rgb, face_bbox)
        phone_detector.draw_hands(frame, hand_lms, is_phone)

        # ── Update Tracker ────────────────────────────────
        tracker.update(ear, mar, pitch, yaw, roll, is_phone)

        # ── FPS ───────────────────────────────────────────
        fps_counter += 1
        if time.time() - fps_timer >= 1.0:
            fps_display = fps_counter
            fps_counter = 0
            fps_timer = time.time()

        # ── Render HUD ────────────────────────────────────
        frame = hud.render(frame, tracker, face_bbox, fps_display)

        # ── Display ───────────────────────────────────────
        cv2.imshow('SleepGuard AI - Smart Classroom Analyzer', frame)

        # ── Keyboard controls ─────────────────────────────
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break
        elif key == ord('d'):
            # Show mini dashboard
            _plot_realtime_history(tracker)
        elif key == ord('r'):
            tracker = StudentTracker('Student_1')
            print('🔄 Stats reset!')
        elif key == ord('s'):
            fname = f'/tmp/sleepguard_{int(time.time())}.png'
            cv2.imwrite(fname, frame)
            print(f'📸 Screenshot saved: {fname}')

    cap.release()
    cv2.destroyAllWindows()

    # ── Final Summary ─────────────────────────────────────
    print('\n' + '='*50)
    print('📋 SESSION SUMMARY')
    print('='*50)
    summary = tracker.session_summary()
    for k, v in summary.items():
        print(f'  {k:20s}: {v}')

    _plot_realtime_history(tracker)
    face_mesh.close()
    face_det.close()


def _run_demo(tracker, hud, duration_sec=30):
    """Demo mode: synthetic data for testing without a camera."""
    import random
    print(f'Running demo for {duration_sec}s...')

    t_start = time.time()
    frame_num = 0
    W, H = hud.fw, hud.fh

    phases = [
        # (duration, ear_mean, ear_std, scenario)
        (8,  0.31, 0.02, 'attentive'),
        (5,  0.19, 0.03, 'drowsy'),
        (4,  0.12, 0.02, 'sleeping'),
        (5,  0.30, 0.02, 'attentive'),
        (4,  0.25, 0.03, 'phone_use'),
        (4,  0.20, 0.04, 'yawning'),
    ]

    phase_idx = 0
    phase_start = t_start

    while time.time() - t_start < duration_sec:
        t = time.time()
        elapsed = t - t_start
        phase_elapsed = t - phase_start

        # Advance phase
        if phase_idx < len(phases) and phase_elapsed > phases[phase_idx][0]:
            phase_idx = min(phase_idx + 1, len(phases)-1)
            phase_start = t

        dur, em, es, scenario = phases[phase_idx]

        ear   = max(0.10, np.random.normal(em, es))
        mar   = 0.70 if scenario == 'yawning' else np.random.normal(0.30, 0.05)
        pitch = np.random.normal(-18 if scenario=='sleeping' else 5, 3)
        yaw   = np.random.normal(0, 4)
        roll  = np.random.normal(25 if scenario=='sleeping' else 3, 2)
        phone = (scenario == 'phone_use') and (random.random() > 0.3)

        tracker.update(ear, mar, pitch, yaw, roll, phone)

        # Create a synthetic frame
        frame = np.zeros((H, W, 3), dtype=np.uint8)
        frame[:] = (25, 25, 40)

        # Draw synthetic face
        cx, cy = W//2, H//2
        cv2.ellipse(frame, (cx, cy), (130, 160), 0, 0, 360, (200,180,150), -1)

        # Eyes
        eye_open = ear > CONFIG['EAR_THRESHOLD']
        ey = int(cy - 40 + pitch * 2)
        for ex in [cx-50, cx+50]:
            cv2.ellipse(frame, (ex, ey), (25, 15 if eye_open else 3),
                        0, 0, 360, (255,255,255), -1)
            if eye_open:
                cv2.circle(frame, (ex, ey), 8, (50,50,200), -1)

        # Mouth
        my = cy + 50
        mh = int(20 * max(mar, 0.2))
        cv2.ellipse(frame, (cx, my), (35, mh), 0, 0, 360, (100,50,50), -1)

        # Scenario label
        cv2.putText(frame, f'DEMO: {scenario.upper()}',
                    (W//2 - 80, H-80), CONFIG['FONT'], 0.7,
                    (200, 200, 200), 1, cv2.LINE_AA)

        frame = hud.render(frame, tracker, (cx-150, cy-180, 300, 360), fps=30)
        cv2.imshow('SleepGuard AI — DEMO MODE', frame)

        if cv2.waitKey(33) & 0xFF in [ord('q'), 27]:
            break
        frame_num += 1

    cv2.destroyAllWindows()
    print('\n✅ Demo complete!')
    print('\n📋 Demo Session Summary:')
    for k, v in tracker.session_summary().items():
        print(f'  {k:25s}: {v}')
    _plot_realtime_history(tracker)


def _plot_realtime_history(tracker):
    """Plot real-time history charts from tracker data."""
    if len(tracker.time_history) < 5:
        print('Not enough data to plot yet.')
        return

    times  = list(tracker.time_history)
    ears   = list(tracker.ear_history)
    scores = list(tracker.score_history)
    sleeps = list(tracker.sleep_history)
    mars   = list(tracker.mar_history)
    pitchs = list(tracker.pitch_history)
    rolls  = list(tracker.roll_history)

    fig, axes = plt.subplots(3, 2, figsize=(14, 9), facecolor='#1a1a2e')
    fig.suptitle(f'📈 {tracker.id} — Real-Time Analysis History',
                 color='white', fontsize=14, fontweight='bold')

    plots = [
        (axes[0,0], ears,   'Eye Aspect Ratio (EAR)', '#44AAFF',
         CONFIG['EAR_THRESHOLD'], 'Closed threshold'),
        (axes[0,1], mars,   'Mouth Aspect Ratio (MAR)', '#FFAA44',
         CONFIG['MAR_THRESHOLD'], 'Yawn threshold'),
        (axes[1,0], scores, 'Attention Score (%)', '#44FF88', 70, 'Alert level'),
        (axes[1,1], sleeps, 'Sleep Probability (%)', '#FF4466', 75, 'Sleep alert'),
        (axes[2,0], pitchs, 'Head Pitch (°)', '#AA44FF',
         CONFIG['HEAD_PITCH_THRESHOLD'], 'Threshold'),
        (axes[2,1], rolls,  'Head Roll (°)', '#FF44AA',
         CONFIG['HEAD_ROLL_THRESHOLD'], 'Threshold'),
    ]

    for ax, data, title, color, thresh, tlabel in plots:
        ax.set_facecolor('#16213e')
        ax.plot(times, data, color=color, linewidth=1.2, alpha=0.9)
        ax.fill_between(times, data, alpha=0.15, color=color)
        ax.axhline(thresh, color='#FF5555', linestyle='--',
                   linewidth=1, alpha=0.7, label=tlabel)
        ax.set_title(title, color='white', fontsize=10)
        ax.tick_params(colors='white')
        ax.set_xlabel('Time (s)', color='#aaaaaa', fontsize=8)
        ax.spines[['top','right']].set_color('#333')
        ax.spines[['bottom','left']].set_color('#555')
        ax.legend(fontsize=7, labelcolor='white',
                  facecolor='#333', edgecolor='#555')

    plt.tight_layout()
    plt.savefig('/tmp/sleepguard_history.png', dpi=120,
                bbox_inches='tight', facecolor='#1a1a2e')
    plt.show()
    print('📊 History chart saved to /tmp/sleepguard_history.png')


print('✅ Main detection loop loaded!')
print('   Run: run_sleepguard() for webcam mode')
print('   Run: run_sleepguard(demo_mode=True) for demo mode')

## 🚀 CELL 10 — Launch SleepGuard AI

In [ ]:
# ============================================================
# CELL 10: LAUNCH — Choose your mode!
# ============================================================
#
#  MODE 1 (Real Webcam):
#       run_sleepguard(camera_index=0)
#
#  MODE 2 (Demo - no camera needed):
#       run_sleepguard(demo_mode=True)
#
#  KEYBOARD CONTROLS while running:
#       Q / ESC  → Quit
#       D        → Show history dashboard
#       R        → Reset statistics
#       S        → Save screenshot
# ============================================================

print('🛡️  SleepGuard AI — Smart Classroom Analyzer')
print('=' * 50)
print('Launching...')
print()

# ▼ CHANGE demo_mode=False TO USE REAL WEBCAM ▼
run_sleepguard(camera_index=0, demo_mode=True)

## 📊 CELL 11 — Classroom Multi-Student Demo Dashboard

In [ ]:
# ============================================================
# CELL 11: Simulate 6-student classroom & show ranking
# ============================================================
import random

dashboard = ClassroomDashboard(num_students=6)

# Simulate 500 frames of data per student
scenarios = [
    {'ear': 0.12, 'mar': 0.80, 'pitch': -20, 'yaw': 5,  'roll': 30, 'phone': 0.0},  # Sleeping
    {'ear': 0.19, 'mar': 0.50, 'pitch': -12, 'yaw': 8,  'roll': 15, 'phone': 0.2},  # Drowsy
    {'ear': 0.30, 'mar': 0.30, 'pitch':   5, 'yaw': 3,  'roll':  5, 'phone': 0.6},  # Phone use
    {'ear': 0.32, 'mar': 0.28, 'pitch':   3, 'yaw': 2,  'roll':  3, 'phone': 0.0},  # Attentive
    {'ear': 0.24, 'mar': 0.60, 'pitch':  -8, 'yaw': 20, 'roll': 10, 'phone': 0.1},  # Distracted
    {'ear': 0.31, 'mar': 0.25, 'pitch':   2, 'yaw': 4,  'roll':  2, 'phone': 0.05}, # Attentive
]

for i, sid in enumerate(dashboard.students.keys()):
    sc = scenarios[i]
    for _ in range(500):
        ear   = max(0.08, np.random.normal(sc['ear'],   0.03))
        mar   = max(0.10, np.random.normal(sc['mar'],   0.05))
        pitch = np.random.normal(sc['pitch'], 3)
        yaw   = np.random.normal(sc['yaw'],   4)
        roll  = np.random.normal(sc['roll'],  3)
        phone = random.random() < sc['phone']
        dashboard.update_student(sid, ear, mar, pitch, yaw, roll, phone)

print('📊 Simulated classroom data — generating dashboard...')
print()

# Print ranking table
print(f'{"Rank":<6}{"Student":<15}{"Attention%":<14}{"Sleep%":<12}{"Yawns":<8}{"Phone":<8}')
print('-' * 60)
for rank, r in enumerate(dashboard.get_ranking(), 1):
    medal = ['🥇','🥈','🥉'][rank-1] if rank <= 3 else f'  #{rank}'
    status = '🔴 SLEEPING' if r['avg_attention'] < 40 else \
             '🟠 DISTRACTED' if r['avg_attention'] < 65 else \
             '🟡 DROWSY' if r['avg_attention'] < 80 else '🟢 OK'
    print(f'{medal:<6}{r["student_id"]:<15}{r["avg_attention"]:<14.1f}'
          f'{r["avg_sleep_prob"]:<12.1f}{r["total_yawns"]:<8}{r["phone_episodes"]:<8}  {status}')

print()
dashboard.plot_dashboard()

## 📋 CELL 12 — Export Session Report to CSV

In [ ]:
# ============================================================
# CELL 12: Export session reports to CSV & HTML
# ============================================================

def export_session_report(dashboard, output_path='/tmp/sleepguard_report'):
    """Export classroom session data to CSV and HTML report."""
    import os
    os.makedirs(output_path, exist_ok=True)

    # ── CSV Summary ───────────────────────────────────────
    rows = [r for r in dashboard.get_ranking()]
    df = pd.DataFrame(rows)
    df['status'] = df['avg_attention'].apply(
        lambda a: 'SLEEPING' if a<40 else 'DISTRACTED' if a<65
                  else 'DROWSY' if a<80 else 'ATTENTIVE'
    )
    csv_path = os.path.join(output_path, 'session_summary.csv')
    df.to_csv(csv_path, index=False)
    print(f'✅ CSV saved: {csv_path}')

    # ── HTML Report ───────────────────────────────────────
    html = '''
    <!DOCTYPE html>
    <html>
    <head>
      <title>SleepGuard AI Report</title>
      <style>
        body { background:#1a1a2e; color:#eee; font-family:Arial; padding:20px; }
        h1 { color:#44aaff; } h2 { color:#aaa; }
        table { border-collapse:collapse; width:100%; }
        th { background:#333; color:#44aaff; padding:10px; text-align:left; }
        td { padding:8px 10px; border-bottom:1px solid #333; }
        tr:hover { background:#222; }
        .SLEEPING   { color:#FF4444; font-weight:bold; }
        .DISTRACTED { color:#FF8800; font-weight:bold; }
        .DROWSY     { color:#FFCC00; font-weight:bold; }
        .ATTENTIVE  { color:#44CC44; font-weight:bold; }
      </style>
    </head>
    <body>
    <h1>🛡️ SleepGuard AI — Session Report</h1>
    <h2>Generated: ''' + time.strftime('%Y-%m-%d %H:%M:%S') + '''</h2>
    <table>
      <tr><th>Rank</th><th>Student</th><th>Attention%</th><th>Sleep%</th>
          <th>Blinks</th><th>Yawns</th><th>Sleep Episodes</th>
          <th>Phone Episodes</th><th>Sleep Time(s)</th><th>Status</th></tr>
    '''
    for rank, row in enumerate(rows, 1):
        s = row['status']
        html += f'''<tr>
          <td>#{rank}</td>
          <td>{row["student_id"]}</td>
          <td>{row["avg_attention"]:.1f}%</td>
          <td>{row["avg_sleep_prob"]:.1f}%</td>
          <td>{row["total_blinks"]}</td>
          <td>{row["total_yawns"]}</td>
          <td>{row["sleep_episodes"]}</td>
          <td>{row["phone_episodes"]}</td>
          <td>{row["total_sleep_sec"]}</td>
          <td class="{s}">{s}</td>
        </tr>'''
    html += '</table></body></html>'

    html_path = os.path.join(output_path, 'session_report.html')
    with open(html_path, 'w') as f:
        f.write(html)
    print(f'✅ HTML report saved: {html_path}')

    print('\n📋 Summary Table:')
    print(df[['student_id','avg_attention','avg_sleep_prob',
              'total_yawns','phone_episodes','status']].to_string(index=False))
    return df


df_report = export_session_report(dashboard)
print('\n✅ All exports complete!')

---
## 🎯 SleepGuard AI — System Architecture

```
┌─────────────────────────────────────────────────────────┐
│              SleepGuard AI Pipeline                      │
│                                                         │
│  📷 Camera Input                                        │
│       │                                                 │
│       ├──► MediaPipe FaceMesh ──► EAR (Eye Closure)     │
│       │                      ──► MAR (Yawn Detection)   │
│       │                      ──► Head Pose (P/Y/R)      │
│       │                                                 │
│       ├──► MediaPipe Hands   ──► Phone Detection        │
│       │                                                 │
│       └──► StudentTracker ──► Attention Score           │
│                           ──► Sleep Probability %       │
│                           ──► Distraction Level         │
│                                                         │
│  📊 HUD Renderer ──► Live Overlay                       │
│  🏆 Dashboard    ──► Class Ranking                      │
│  📋 Reporter     ──► CSV / HTML Export                  │
└─────────────────────────────────────────────────────────┘
```

### Key Algorithms:
| Algorithm | Method | Threshold |
|-----------|--------|-----------|
| Eye Closure | Eye Aspect Ratio (EAR) | < 0.22 |
| Yawn Detection | Mouth Aspect Ratio (MAR) | > 0.65 |
| Head Pose | solvePnP + Euler angles | Pitch>15°, Yaw>30°, Roll>20° |
| Phone Use | Hand landmark heuristic | Score > 0.55 |
| Attention Score | Weighted combination | 0–100% |
| Sleep Probability | Eye+Head+Yawn weighted | 0–100% |